In [1]:
from pathlib import Path
import json
from collections import Counter, defaultdict
import os

In [2]:
def read_jsonl(path:Path):
    data = []
    with path.open("r") as f:
        for line in f:
            data.append(json.loads(line))

    return data

In [5]:
data_paths = [
    ("topic_raw", Path("../results/LLM_as_judge/label2doc_desc2label/topic/raw/desc2label_responses_descriptors2label.jsonl")),
    ("topic_harmonized", Path("../results/LLM_as_judge/label2doc_desc2label/topic/harmonized/desc2label_responses_descriptors2label.jsonl")),
    ("format_raw", Path("../results/LLM_as_judge/label2doc_desc2label/format/raw/desc2label_responses_descriptors2label.jsonl")),
    ("format_harmonized", Path("../results/LLM_as_judge/label2doc_desc2label/format/harmonized/desc2label_responses_descriptors2label.jsonl")),
]

In [6]:
for data_name, path in data_paths:
    data = read_jsonl(path)
    results = defaultdict(Counter)
    for doc in data:
        if "topic" in data_name:
            label = doc["topic"]
        else:
            label = doc["format"]
        response = doc["parsed_response"]
        results[label].update([response])
    print("========")
    print(data_name)
    print("========")
    for k, v in results.items():
        print(k)
        most = v.most_common()
        total = sum(v.values()) 
        print([(k, v, round(v/total,2)) for k, v in most])
        print()
    del data

FileNotFoundError: [Errno 2] No such file or directory: '../results/LLM_as_judge/label2doc_desc2label/topic/raw/desc2label_responses_descriptors2label.jsonl'

In [5]:
def read_json(path:Path):
    with path.open("r") as f:
        return json.loads(f.read())

In [6]:
paths = [
("harmonized_formats", Path("../results/faiss/pipeline/formats/harmonized/pipeline_summary.json")),
("raw_format", Path("../results/faiss/pipeline/formats/raw/pipeline_summary.json")),
("harmonized_topics", Path("../results/faiss/pipeline/topics/harmonized/pipeline_summary.json")),
("raw_topics", Path("../results/faiss/pipeline/topics/raw/pipeline_summary.json"))
]

data = {}
for name, path in paths:
    data[name] = read_json(path)

In [7]:
results = []
for name in data:
    for query in data[name]["runs"]:
        if "harmonized" in name:
            descriptor_type = "harmonized"
        else:
            descriptor_type = "raw"
        if "formats" in name:
            label_type = "format"
        else:
            label_type = "topic"
        query_head = query["query"].split(";")[0]
        search_hits = query["search"]["results"]
        LLM_approved_descriptors = query["descriptor_judge"]["selected_descriptors"]
        document_hits = query["documents"]["selected"]
        document_judgements = query["document_judge"]["labels"]
        results.append({
            "descriptor_type": descriptor_type,
            "label_type": label_type,
            "label": query_head,
            "search_hits": search_hits,
            "LLM_approved_descriptors": LLM_approved_descriptors,
            "documents_found": document_hits,
            "document_judgements": document_judgements
        })

In [17]:
for res in results:
    if res["descriptor_type"] =="harmonized" and res["label_type"] == "format":
        print(res)

{'descriptor_type': 'harmonized', 'label_type': 'format', 'label': 'About (Org.)', 'search_hits': 407, 'LLM_approved_descriptors': 32, 'documents_found': 240, 'document_judgements': {'partial': 27, 'yes': 110, 'no': 103}}
{'descriptor_type': 'harmonized', 'label_type': 'format', 'label': 'About (Personal)', 'search_hits': 990, 'LLM_approved_descriptors': 176, 'documents_found': 6981, 'document_judgements': {'no': 3522, 'yes': 2838, 'partial': 618, 'invalid': 3}}
{'descriptor_type': 'harmonized', 'label_type': 'format', 'label': 'Academic Writing', 'search_hits': 1677, 'LLM_approved_descriptors': 146, 'documents_found': 4405, 'document_judgements': {'no': 2369, 'partial': 183, 'yes': 1853}}
{'descriptor_type': 'harmonized', 'label_type': 'format', 'label': 'Audio Transcript', 'search_hits': 355, 'LLM_approved_descriptors': 54, 'documents_found': 568, 'document_judgements': {'yes': 265, 'no': 295, 'partial': 8}}
{'descriptor_type': 'harmonized', 'label_type': 'format', 'label': 'Comment 

In [11]:
def print_precision_and_recall_per_label(data_type:str, acceptable_labels:set):

    descriptor_types = ("harmonized", "raw")
    label_types = ("formats", "topics")
    
    data_type = "full"
    
    if data_type == "filtered":
        verified_formats_path = Path("../data/weborganizer/LLM_verified_formats_edu.jsonl")
        verified_topics_path = Path("../data/weborganizer/LLM_verified_topics_edu.jsonl")
        verified_formats = read_jsonl(verified_formats_path)
        verified_topics = read_jsonl(verified_topics_path)
    elif data_type == "full":
        data_path = Path("../data/weborganizer/topic_format_edu.jsonl")
        verified_formats = read_jsonl(data_path)
        verified_topics = verified_formats
    else:
        print(f"Invalid data_type: {data_type}")
        return
    
    def normalize_label(label):
        if label == "About (Personal)":
            return "About (Pers.)"
        elif label == "Science & Technology":
            return "Science & Tech."
        elif label == "Software Development":
            return "Software Dev."
        else:
            return label.strip()
    
    
    def find_by_label(verified_docs, label, label_type):
        label = normalize_label(label)
        doc_ids = []
    
        for doc in verified_docs:
            if label_type == "formats":
                doc_label = normalize_label(doc["format"])
            elif label_type == "topics":
                doc_label = normalize_label(doc["topic"])
            else:
                raise ValueError(f"Unknown label_type: {label_type}")
    
            if doc_label == label:
                doc_ids.append(doc["doc_id"])
    
        assert doc_ids, f"No documents found for label={label!r}, label_type={label_type!r}"
        return doc_ids
    
    
    verified_formats_by_id = {doc["doc_id"]: doc for doc in verified_formats}
    verified_topics_by_id = {doc["doc_id"]: doc for doc in verified_topics}
    
    
    for desc_type in descriptor_types:
        for l_type in label_types:
            print("===================")
            print("|",l_type, desc_type,"|")
            print("===================")

            global_model_counter = Counter()
    
            dir_path = Path(f"../results/faiss/pipeline/full/{l_type}/{desc_type}/")
    
            for res_dir in dir_path.iterdir():
                if not res_dir.is_dir():
                    continue
    
                file = res_dir / "final_results.jsonl"
                if file.exists():
                    query_results = read_jsonl(file)
                else:
                    query_results = None
                if not query_results:
                    print(file)
                    print("True positives:", 0)
                    print("False positives:", 0)
                    print("Precision:", 0)
                    print("Recall:", 0)
                    print()
                    continue
    
                query_label = None
                true_positive_doc_ids = set()
                false_positive_doc_ids = set()
                accepted_doc_ids = set()
                model_judgments = Counter()
    
                for row in query_results:
                    query_label = normalize_label(row["query"].split(";")[0])

                    model_judgment = row["label"].lower().strip()
                    if model_judgment == "invalid":
                        model_judgment = "no"
                    model_judgments.update([model_judgment])

                    if model_judgment not in acceptable_labels:
                        continue
    
                    found_doc_id = row["doc_id"]
                    accepted_doc_ids.add(found_doc_id)
    
                    if l_type == "topics":
                        verified_doc = verified_topics_by_id.get(found_doc_id)
                        verified_label = normalize_label(verified_doc["topic"])
                    elif l_type == "formats":
                        verified_doc = verified_formats_by_id.get(found_doc_id)
                        verified_label = normalize_label(verified_doc["format"])
                    else:
                        raise ValueError(f"Unknown label type: {l_type}")
    
                    row["WO_label"] = verified_label
    
                    if verified_label == query_label:
                        true_positive_doc_ids.add(found_doc_id)
                    else:
                        false_positive_doc_ids.add(found_doc_id)
    
                if l_type == "formats":
                    all_true_doc_ids = set(find_by_label(verified_formats, query_label, l_type))
                elif l_type == "topics":
                    all_true_doc_ids = set(find_by_label(verified_topics, query_label, l_type))
                else:
                    raise ValueError(f"Unknown label type: {l_type}")
    
                false_negative_doc_ids = all_true_doc_ids - true_positive_doc_ids
    
                tp = len(true_positive_doc_ids)
                fp = len(false_positive_doc_ids)
                fn = len(false_negative_doc_ids)
    
                precision = tp / (tp + fp) if tp + fp > 0 else 0
                recall = tp / len(all_true_doc_ids) if all_true_doc_ids else 0

                global_model_counter.update(model_judgments)
    
                print(query_label)
                print("True positives:", tp)
                print("False positives:", fp)
                print("False negatives:", fn)
                print("Precision:", round(precision, 2))
                print("Recall:", round(recall, 2))
                print("Model judgments:")
                for k,v in model_judgments.items():
                    print(f"{k}: {v} ({round(v/model_judgments.total(),2)})")
                print()
            print(global_model_counter.most_common())
            print((global_model_counter["yes"]+global_model_counter["partial"])/global_model_counter.total())
            print()

print_precision_and_recall_per_label(data_type="full", acceptable_labels={"yes", "partial", "no"})

| formats harmonized |
Tutorial
True positives: 2172
False positives: 3052
False negatives: 2426
Precision: 0.42
Recall: 0.47
Model judgments:
yes: 2876 (0.55)
no: 2070 (0.4)
partial: 278 (0.05)

Creative Writing
True positives: 417
False positives: 4943
False negatives: 778
Precision: 0.08
Recall: 0.35
Model judgments:
no: 3408 (0.64)
yes: 1694 (0.32)
partial: 258 (0.05)

News (Org.)
True positives: 1829
False positives: 3167
False negatives: 4484
Precision: 0.37
Recall: 0.29
Model judgments:
no: 1327 (0.27)
yes: 3523 (0.71)
partial: 146 (0.03)

About (Org.)
True positives: 78
False positives: 264
False negatives: 3352
Precision: 0.23
Recall: 0.02
Model judgments:
no: 165 (0.48)
yes: 136 (0.4)
partial: 41 (0.12)

Structured Data
True positives: 70
False positives: 122
False negatives: 1588
Precision: 0.36
Recall: 0.04
Model judgments:
yes: 106 (0.55)
no: 69 (0.36)
partial: 17 (0.09)

User Review
True positives: 249
False positives: 719
False negatives: 1624
Precision: 0.26
Recall: 0.1